<a href="https://colab.research.google.com/github/aashrencs24/AI_SEM5_1BF24CS006/blob/main/Week6_1BF24CS006_Unification_first_order.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
def is_variable(x):
    """Check whether a term is a variable."""
    return isinstance(x, str) and x.startswith("?")


def occurs_check(var, term, subst):
    """Check whether a variable occurs in a term."""
    term = apply_substitution(term, subst)

    if var == term:
        return True

    if isinstance(term, tuple):
        return any(occurs_check(var, arg, subst) for arg in term[1:])

    return False


def apply_substitution(term, subst):
    """Apply substitutions to a term."""
    if is_variable(term) and term in subst:
        return apply_substitution(subst[term], subst)

    if isinstance(term, tuple):
        return (term[0],) + tuple(
            apply_substitution(arg, subst) for arg in term[1:]
        )

    return term


def unify(term1, term2, subst=None):
    """Unify two terms and return the substitution or None."""

    if subst is None:
        subst = {}

    # Apply existing substitutions
    term1 = apply_substitution(term1, subst)
    term2 = apply_substitution(term2, subst)

    # Step 1: If terms are identical, return NIL
    if term1 == term2:
        return subst

    # Step 2: If the first term is a variable
    if is_variable(term1):
        if occurs_check(term1, term2, subst):
            return None

        subst[term1] = term2
        return subst

    # Step 3: If the second term is a variable
    if is_variable(term2):
        if occurs_check(term2, term1, subst):
            return None

        subst[term2] = term1
        return subst

    # Step 4: Both terms must be compound terms
    if isinstance(term1, tuple) and isinstance(term2, tuple):

        # Check predicate/function symbols
        if term1[0] != term2[0]:
            return None

        # Check number of arguments
        if len(term1) != len(term2):
            return None

        # Step 5: Unify corresponding arguments
        for arg1, arg2 in zip(term1[1:], term2[1:]):
            subst = unify(arg1, arg2, subst)

            if subst is None:
                return None

        return subst

    # Step 6: Terms cannot be unified
    return None


# Helper to safely parse user input into Python tuples
import ast

def parse_term(input_str):
    """Parses a string input like "('P', '?x', 'a')" into a tuple."""
    try:
        parsed = ast.literal_eval(input_str.strip())
        if isinstance(parsed, tuple) or isinstance(parsed, str):
            return parsed
        else:
            print("Warning: Input parsed, but it is not a tuple or a string.")
            return parsed
    except Exception as e:
        print(f"Error parsing input: {e}. Please use valid Python literal syntax.")
        return None

# Interactive execution
print("=== Interactive Unification ===")
print("Please enter terms as Python tuples, e.g., ('P', '?x', 'a') or ('P', 'b', 'a')")
input_1 = input("Enter Term 1: ")
input_2 = input("Enter Term 2: ")

t1 = parse_term(input_1)
t2 = parse_term(input_2)

if t1 is not None and t2 is not None:
    print(f"\nUnifying:\n  Term 1: {t1}\n  Term 2: {t2}")
    result = unify(t1, t2)
    if result is None:
        print("RESULT: FAILURE (Terms cannot be unified)")
    elif not result:
        print("RESULT: NIL (No substitution required)")
    else:
        print("RESULT: Substitution =", result)


=== Interactive Unification ===
Please enter terms as Python tuples, e.g., ('P', '?x', 'a') or ('P', 'b', 'a')
Enter Term 1: ('P','?x','b')
Enter Term 2: ('P','a','?x')

Unifying:
  Term 1: ('P', '?x', 'b')
  Term 2: ('P', 'a', '?x')
RESULT: FAILURE (Terms cannot be unified)
